# Main processing file
---
### author: Juju Fars
### date: 2024-12

Turns the raw per-session behavioural (psychophysics) and MRS export files into the
"refined" summary tables the other `analysis_*.ipynb` notebooks read from `Data/Refined`.

Runs the same pipeline twice: once for the patient group (writes to `Data/Refined`,
including the MRS data), then again for the control group ("Control Data" section
below, writes to `Data/Controls/Refined`, no MRS data). For each group it:
1. Reads and concatenates the raw per-subject Excel files for every task (cut-off
   frequency, contrast sensitivity, interocular imbalance, dichoptic noise, stereoacuity,
   Snellen/crowded acuity) via `bv.process_files_total`.
2. Derives extra measures (sensitivity, logBP, cut-off in cpd, visual angle/logMAR, etc.)
   and pre/post training differences for each task.
3. Fits contrast-sensitivity and interocular-imbalance curves and computes the
   Area Between Curves (ABC) as a training-effect measure.
4. Saves everything to the numbered `bv_*.xlsx` (and, for patients, `mrs_results_main.xlsx`)
   files that downstream notebooks expect.


In [ ]:
import importlib
import import_packages
importlib.reload(import_packages) # so I can add new packages in import_packages.py without restarting the kernel
from import_packages import *
print("Packages (re)loaded successfully!")


### Set up the paths for the main processing

In [ ]:
# Set Up the paths 
ori = os.getcwd()

path_data_raw = os.path.abspath(os.path.join(ori, "..", "Data", "Raw")) # Start at the notebook directory, go up one level, go into Data/Raw
path_data_ref = os.path.abspath(os.path.join(ori, "..", "Data", "Refined"))

### Read the raw behavioural data 

In [ ]:
pattern_files = ["cutoff", "interleaved", "binoc", "dichop", "asteroid", "snellen", "crowded"]
do_refine=False
remove=None #e.g. ['sub_007']

# Dictionaries to store the DataFrames
df_dict = {}
df_thresholds_dict = {}

# Loop through the pattern_file list
for pattern in pattern_files:
    df_key = f"df_{pattern[:2]}"  # Use the first two characters of the pattern as a key
    thresholds_key = f"{df_key}_thresholds"
    
    # Call the function and store results
    df_dict[df_key], df_thresholds_dict[thresholds_key] = bv.process_files_total(
        path_data_raw, pattern_file=pattern, refined=do_refine, remove_dirs=remove
    )


In [ ]:
# df = df_dict['df_cr'].copy()
df = df_dict['df_sn'].copy()

df = df.sort_values(["ID", "Trial"]).reset_index(drop=True)

In [ ]:


def compute_threshold(subdf):
    # identify reversal points (increase in Reversal)
    subdf = subdf.copy()
    subdf["is_reversal"] = subdf["Reversal"].diff() > 0

    reversal_values = subdf.loc[subdf["is_reversal"], "opto_size_pixel"]

    # # no reversals at all
    # if len(reversal_values) == 0:
    threshold = subdf["opto_size_pixel"].iloc[-1]
    n_reversals = 0

    # # remove first reversal
    # reversal_values = reversal_values.iloc[1:]

    # # still no reversals after removing first
    # if len(reversal_values) == 0:
    #     threshold = subdf["opto_size_pixels"].iloc[-1]
    #     n_reversals = 1

    # # fewer than 6 reversals
    # elif len(reversal_values) < 6:
    #     threshold = np.mean(
    #         list(reversal_values) + [subdf["opto_size_pixels"].iloc[-1]]
    #     )
    #     n_reversals = len(reversal_values)

    # # 6 or more reversals
    # else:
    #     threshold = reversal_values.iloc[-6:].mean()
    #     n_reversals = 6

    return threshold, n_reversals


In [ ]:
rows = []

group_cols = ["ID", "Eye", "stage", "Task", "Date_time"]

for keys, subdf in df.groupby(group_cols):
    threshold, n_reversals = compute_threshold(subdf)

    last_row = subdf.iloc[-1]

    rows.append({
        "Task": last_row["Task"],
        "Date_time": last_row["Date_time"],
        "Trial": last_row["Trial"],
        "Reversal": n_reversals,
        "AMB_eye": "MEAN",
        "Eye_tested": "REVERSAL",
        "opto_size_pixel": threshold,
        # "Orientation_center": 0,
        # "Orientation_left": 0,
        # "Orientation_top": 0,
        # "Orientation_bottom": 0,
        # "Orientation_right": 0,
        "Orientation": 0,
        "Position": 0,
        "Key": "Incomplete_reversals",
        "Success": "end_of_table",
        "RT": 0.0,
        "ID": last_row["ID"],
        "Eye": last_row["Eye"],
        "stage": last_row["stage"],
        "amb": last_row["amb"],
    })


threshold_df = pd.DataFrame(rows)


In [ ]:
df_thresholds_dict['df_sn_thresholds'] = threshold_df

### Useful operations on raw data
Derive extra columns (Sensitivity, logBP, cut-off in cpd, visual angle/logMAR), compute
pre/post-training differences per task, then fit curves and compute the ABC training-effect
measure for contrast sensitivity and interocular imbalance.


In [ ]:

df_thresholds_dict['df_cu_thresholds']['Frequency_cpd'] = df_thresholds_dict['df_cu_thresholds']['Frequency_cpp'] * 72
df_dict['df_in']['Sensitivity'] = df_dict['df_in']['Contrast'].apply(lambda x: 1/x)
df_thresholds_dict['df_in_thresholds']['Sensitivity'] = df_thresholds_dict['df_in_thresholds']['Contrast'].apply(lambda x: 1/x)
df_thresholds_dict['df_bi_thresholds']['logBP'] = df_thresholds_dict['df_bi_thresholds']['Contrast_FE'].apply(bv.log_bp)
df_thresholds_dict['df_di_thresholds']['Noise100'] = df_thresholds_dict['df_di_thresholds']['Noise'] * 100
df_thresholds_dict['df_di_thresholds']['Frequency_cpd'] = df_thresholds_dict['df_di_thresholds']['Frequency_cpp'] * 72
df_thresholds_dict['df_as_thresholds'] = df_dict['df_as'][df_dict['df_as']['trialtime'] == 0]
df_thresholds_dict['df_sn_thresholds'] = bv.calculate_visual_metrics(df_thresholds_dict['df_sn_thresholds'], screen_width_mm=700, screen_height_mm=392, screen_resolution_width=1920, 
                                               screen_resolution_height=1040, distance_mm=2500, opto_size_pixel_column='opto_size_pixel')
df_thresholds_dict['df_cr_thresholds'] = bv.calculate_visual_metrics(df_thresholds_dict['df_cr_thresholds'], screen_width_mm=700, screen_height_mm=392, screen_resolution_width=1920, 
                                               screen_resolution_height=1040, distance_mm=2500, opto_size_pixel_column='opto_size_pixels')


# List of tuples with each dictionary's DataFrame, pivot parameters, and column names for difference calculation
df_info = [
    (df_thresholds_dict['df_cu_thresholds'], ['ID', 'amb'], 'Frequency_cpd', 'cutoff', 'Diff'),
    (df_thresholds_dict['df_in_thresholds'], ['ID', 'Staircase', 'amb'], 'Sensitivity', 'contrast_sensitivity', 'Difference'),
    (df_thresholds_dict['df_bi_thresholds'], ['ID', 'Staircase', 'amb'], 'logBP', 'binoc', 'Difference'),
    (df_thresholds_dict['df_di_thresholds'], ['ID', 'amb'], 'Noise', 'dichop', 'Difference'),
    (df_thresholds_dict['df_as_thresholds'], ['ID'], 'disparitylog', 'asteroid', 'Diff', 3),  # For 'pretraining' fill
    (df_thresholds_dict['df_sn_thresholds'], ['ID', 'amb'], 'logMAR', 'snellen', 'Difference'),
    (df_thresholds_dict['df_cr_thresholds'], ['ID', 'amb'], 'logMAR', 'crowded', 'Difference')
]

# Initialize a dictionary to store the results
df_diff = {}

# Loop through the DataFrame information to perform the pivot and difference calculation
for df, index_cols, value_col, name, diff_col, *fill_value in df_info:
    # Perform the pivot operation
    df_pivot = df.pivot_table(index=index_cols, columns='stage', values=value_col)
    
    # If a fill value for NaN is specified (like in df_as_thresholds), fill it
    if fill_value:
        fill_value = fill_value[0]
        df_pivot['pretraining'] = df_pivot['pretraining'].fillna(fill_value)

    # Calculate the difference between posttraining and pretraining (adjust column names as necessary)
    df_pivot[diff_col] = df_pivot['posttraining'] - df_pivot['pretraining']
    
    # Reset index and save the result to the dictionary
    df_diff[name] = df_pivot.reset_index()



In [ ]:
df_thresholds_dict['df_bi_thresholds']['Frequency_dva'] = round(df_thresholds_dict['df_bi_thresholds']['Frequency_dva'], 1)

In [ ]:
# Calculate Area Below Curve (ABC) for interleaved contrast sensitivity and interocular imblance curves 
abc_df_in = bv.calculate_auc_difference_updated(df_thresholds_dict['df_in_thresholds'], x_col='Frequency_dva', y_col='Sensitivity', fit_type='logparabola', range_mode = 'fixed', fixed_range=(0, 50), plot='../Data/Plots/contast_fit')
# 'weibull' | 'linear' | 'logparabola' | 'barten' | 'daly'

abc_df_bi = bv.calculate_auc_difference_updated(df_thresholds_dict['df_bi_thresholds'], x_col='Frequency_dva', y_col='logBP', fit_type='linear', plot='../Data/Plots/imbalance_fit')

# Combine ABC into a dictionary
ABC_dict = {"abc_df_in": abc_df_in,"abc_df_bi": abc_df_bi}

# Combine data
full_df = bv.process_threshold_data(df_thresholds_dict['df_cu_thresholds'], df_thresholds_dict['df_in_thresholds'], df_thresholds_dict['df_bi_thresholds'], df_thresholds_dict['df_di_thresholds'], df_thresholds_dict['df_as_thresholds'], df_thresholds_dict['df_sn_thresholds'], df_thresholds_dict['df_cr_thresholds'])
full_df_diff = bv.calculate_training_differences(full_df)

### Read the MRS data

In [ ]:
# Read the raw MRS data 
MRS_diff_df = pd.read_csv(path_data_raw+'/combined_summary_diff.csv')
MRS_qc_df = pd.read_csv(path_data_raw+'/combined_qc_diff.csv')
MRS_tf_df = pd.read_csv(path_data_raw+'/combined_tissue_fraction.csv')
# MRS participant numbers come from BIDS folder names (sub-001 -> '001'); use the published 'P01' format
for _df in (MRS_diff_df, MRS_qc_df, MRS_tf_df):
    _df['participant_number'] = _df['participant_number'].map(lambda x: bv.format_participant_id(x, 'P'))


### Perform operations on MRS data

In [ ]:
# refine diff df
columns_to_delete = ['SNR', 'FWHM']
for col in columns_to_delete:
    if col in MRS_diff_df.columns:
        del MRS_diff_df[col]

metabs = ['GABA+sysMM', 'Glu+Gln+GSH'] # metabs of interest
locations = ['M1', 'V1'] # locations of interest

metab_diff, df_metab = mrs.process_mrs_diff_data(MRS_diff_df, metabs, locations)

columns_to_keep = ['Metab', 'mMol/kg', 'participant_number', 'session', 'location']
metab_diff_subset = metab_diff[columns_to_keep]
metab_diff_subset = metab_diff_subset.pivot_table(index=['participant_number', 'location', 'Metab'],columns='session', values='mMol/kg').reset_index()
metab_diff_subset['diff'] = metab_diff_subset['post'] - metab_diff_subset['pre']
metab_diff_subset.rename(columns={'participant_number': 'ID'}, inplace=True)
metab_diff_subset["Metab"] = metab_diff_subset["Metab"].replace({'GABA+sysMM':'GABA+', 'Glu+Gln+GSH':'Glx'})

qc_diff = MRS_qc_df
#qc_diff = MRS_qc_df.pivot_table(index=['participant_number', 'location', 'Metab'],columns='session', values=['SNR', 'FWHM']).reset_index()
#qc_diff.columns = ['_'.join(filter(None, col)).strip() for col in qc_diff.columns.to_flat_index()]
qc_diff.rename(columns={'participant_number': 'ID'}, inplace=True)

tf_diff = MRS_tf_df
#tf_diff = MRS_tf_df.pivot_table(index=['participant_number', 'location'],columns='session', values=['CSF', 'GM', 'WM']).reset_index()
#tf_diff.columns = ['_'.join(filter(None, col)).strip() for col in tf_diff.columns.to_flat_index()]
tf_diff.rename(columns={'participant_number': 'ID'}, inplace=True)

# Combine MRS dfs into a dictionary
results_MRS = {"metab": metab_diff_subset,"quality_control": qc_diff, "tissue_fraction":tf_diff}


### Save the refined results

In [ ]:
# Save all the refined data 

# save dicts
dict_list = [
    (df_dict, "bv_1_results_trials.xlsx"),
    (df_thresholds_dict, "bv_2_results_thresholds.xlsx"),
    (df_diff, "bv_3_results_Diff.xlsx"), 
    (ABC_dict, "bv_4_results_ABC_diff.xlsx"), 
    (results_MRS, "mrs_results_main.xlsx")
    # Add more dictionaries and filenames as needed
]

# Loop through the list and save each dictionary as an Excel file
for data_dict, filename in dict_list:
    file_path = os.path.join(path_data_ref, filename)
    # Save the DataFrame dictionary to the Excel file
    with pd.ExcelWriter(file_path) as writer:
        for key, df in data_dict.items():
            # Use the dictionary key as the sheet name
            sheet_name = key[:31]  # Truncate to Excel's 31-character limit
            df.to_excel(writer, sheet_name=sheet_name, index=False)
    print(f"Saved {filename} with sheets: {list(data_dict.keys())}")

# save dfs
full_df.to_excel(os.path.join(path_data_ref, 'bv_5_combined.xlsx'), index=False, sheet_name="Sheet1")
full_df_diff.to_excel(os.path.join(path_data_ref, 'bv_5_combined_diff.xlsx'), index=False, sheet_name="Sheet1")# this last one might be useless


# Control Data

In [ ]:
# Set Up the paths 
ori = os.getcwd()
path_data_raw = os.path.abspath(os.path.join(ori, "..", "Data", "Controls", "Raw")) # Start at the notebook directory, go up one level, go into Data/Raw
path_data_ref = os.path.abspath(os.path.join(ori, "..", "Data", "Controls", "Refined"))

In [ ]:
pattern_files = ["cutoff", "interleaved", "binoc", "dichop", "asteroid", "snellen", "crowded"]
do_refine=True
remove=None #e.g. ['sub_007']

# Dictionaries to store the DataFrames
df_dict = {}
df_thresholds_dict = {}

# Loop through the pattern_file list
for pattern in pattern_files:
    df_key = f"df_{pattern[:2]}"  # Use the first two characters of the pattern as a key
    thresholds_key = f"{df_key}_thresholds"
    
    # Call the function and store results
    df_dict[df_key], df_thresholds_dict[thresholds_key] = bv.process_files_total(
        path_data_raw, pattern_file=pattern, refined=do_refine, remove_dirs=remove, folder_head='con_*', id_prefix='C'
    )


In [ ]:
#

df_thresholds_dict['df_cu_thresholds']['Frequency_cpd'] = df_thresholds_dict['df_cu_thresholds']['Frequency_cpp'] * 72
df_dict['df_in']['Sensitivity'] = df_dict['df_in']['Contrast'].apply(lambda x: 1/x)
df_thresholds_dict['df_in_thresholds']['Sensitivity'] = df_thresholds_dict['df_in_thresholds']['Contrast'].apply(lambda x: 1/x)
df_thresholds_dict['df_bi_thresholds']['logBP'] = (df_thresholds_dict['df_bi_thresholds']['Contrast_FE']/df_thresholds_dict['df_bi_thresholds']['Contrast_AE']).apply(bv.log_bp)
df_thresholds_dict['df_di_thresholds']['Noise100'] = df_thresholds_dict['df_di_thresholds']['Noise'] * 100
df_thresholds_dict['df_di_thresholds']['Frequency_cpd'] = df_thresholds_dict['df_di_thresholds']['Frequency_cpp'] * 72
df_thresholds_dict['df_as_thresholds'] = df_dict['df_as'][df_dict['df_as']['trialtime'] == 0]
df_thresholds_dict['df_sn_thresholds'] = bv.calculate_visual_metrics(df_thresholds_dict['df_sn_thresholds'], screen_width_mm=700, screen_height_mm=392, screen_resolution_width=1920, 
                                               screen_resolution_height=1040, distance_mm=2500, opto_size_pixel_column='opto_size_pixel')
df_thresholds_dict['df_cr_thresholds'] = bv.calculate_visual_metrics(df_thresholds_dict['df_cr_thresholds'], screen_width_mm=700, screen_height_mm=392, screen_resolution_width=1920, 
                                               screen_resolution_height=1040, distance_mm=2500, opto_size_pixel_column='opto_size_pixels')


# List of tuples with each dictionary's DataFrame, pivot parameters, and column names for difference calculation
df_info = [
    (df_thresholds_dict['df_cu_thresholds'], ['ID', 'amb'], 'Frequency_cpd', 'cutoff', 'Diff'),
    (df_thresholds_dict['df_in_thresholds'], ['ID', 'Staircase', 'amb'], 'Sensitivity', 'contrast_sensitivity', 'Difference'),
    (df_thresholds_dict['df_bi_thresholds'], ['ID', 'Staircase', 'amb'], 'logBP', 'binoc', 'Difference'),
    (df_thresholds_dict['df_di_thresholds'], ['ID', 'amb'], 'Noise', 'dichop', 'Difference'),
    (df_thresholds_dict['df_as_thresholds'], ['ID'], 'disparitylog', 'asteroid', 'Diff'),  # For 'pretraining' fill
    (df_thresholds_dict['df_sn_thresholds'], ['ID', 'amb'], 'logMAR', 'snellen', 'Difference'),
    (df_thresholds_dict['df_cr_thresholds'], ['ID', 'amb'], 'logMAR', 'crowded', 'Difference')
]

# Initialize a dictionary to store the results
df_diff = {}

# Loop through the DataFrame information to perform the pivot and difference calculation
for df, index_cols, value_col, name, diff_col, *fill_value in df_info:
    # Perform the pivot operation
    df_pivot = df.pivot_table(index=index_cols, columns='stage', values=value_col)
    
    # If a fill value for NaN is specified (like in df_as_thresholds), fill it
    if fill_value:
        fill_value = fill_value[0]
        df_pivot['pretraining'] = df_pivot['pretraining'].fillna(fill_value)

    # Calculate the difference between posttraining and pretraining (adjust column names as necessary)
    df_pivot[diff_col] = df_pivot['posttraining'] - df_pivot['pretraining']
    
    # Reset index and save the result to the dictionary
    df_diff[name] = df_pivot.reset_index()

df_thresholds_dict['df_bi_thresholds']['Frequency_dva'] = round(df_thresholds_dict['df_bi_thresholds']['Frequency_dva'], 1)

# Calculate Area Below Curve (ABC) for interleaved contrast sensitivity and interocular imblance curves 
abc_df_in = bv.calculate_auc_difference_updated(df_thresholds_dict['df_in_thresholds'], x_col='Frequency_dva', y_col='Sensitivity', fit_type='logparabola', range_mode = 'fixed', fixed_range=(0, 50))
#abc_df_in = abc_df_in.set_index(['ID', 'amb'])['ABC'].to_dict()
abc_df_bi = bv.calculate_auc_difference_updated(df_thresholds_dict['df_bi_thresholds'], x_col='Frequency_dva', y_col='logBP', fit_type='linear')
#abc_df_bi = abc_df_bi.set_index(['ID', 'amb'])['ABC'].to_dict()
# Combine ABC into a dictionary
ABC_dict = {"abc_df_in": abc_df_in,"abc_df_bi": abc_df_bi}

# Combine data
full_df = bv.process_threshold_data(df_thresholds_dict['df_cu_thresholds'], df_thresholds_dict['df_in_thresholds'], df_thresholds_dict['df_bi_thresholds'], df_thresholds_dict['df_di_thresholds'], df_thresholds_dict['df_as_thresholds'], df_thresholds_dict['df_sn_thresholds'], df_thresholds_dict['df_cr_thresholds'])
full_df_diff = bv.calculate_training_differences(full_df)


In [ ]:
# Save all the refined data 

# save dicts
dict_list = [
    (df_dict, "bv_1_results_trials.xlsx"),
    (df_thresholds_dict, "bv_2_results_thresholds.xlsx"),
    (df_diff, "bv_3_results_Diff.xlsx"), 
    (ABC_dict, "bv_4_results_ABC_diff.xlsx"), 
    # Add more dictionaries and filenames as needed
]

# Loop through the list and save each dictionary as an Excel file
for data_dict, filename in dict_list:
    file_path = os.path.join(path_data_ref, filename)
    # Save the DataFrame dictionary to the Excel file
    with pd.ExcelWriter(file_path) as writer:
        for key, df in data_dict.items():
            # Use the dictionary key as the sheet name
            sheet_name = key[:31]  # Truncate to Excel's 31-character limit
            df.to_excel(writer, sheet_name=sheet_name, index=False)
    print(f"Saved {filename} with sheets: {list(data_dict.keys())}")

# save dfs
full_df.to_excel(os.path.join(path_data_ref, 'bv_5_combined.xlsx'), index=False, sheet_name="Sheet1")
full_df_diff.to_excel(os.path.join(path_data_ref, 'bv_5_combined_diff.xlsx'), index=False, sheet_name="Sheet1")# this last one might be useless
